# 10 · Prefetch, push consumers, and competing workers

**Level:** advanced · **Time:** 35 minutes · **Prerequisites:** 00 and 02.

Observe how prefetch limits unacknowledged push deliveries, then run two real workers with different speeds. Each worker owns its connection and channel. Pika connections are not shared between threads.

**Prediction:** with eight queued messages and no acknowledgements, prefetch 1 permits one delivery; prefetch 4 permits four. `basic_get` is a polling API and does not demonstrate prefetch, so this notebook uses `basic_consume` callbacks.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course" / "rabbit.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.rabbit import connect, unique_name, wait_for_message, broker_url
import pika
print("Course root:", ROOT)


## Step 1 — Measure the unacknowledged delivery window

The callback deliberately records deliveries without acknowledging. We pump the connection for up to two seconds, then cancel and clean up. The messages are disposable numbers created by this experiment.


In [ ]:
import time

def observe_window(prefetch):
    queue_name = unique_name(f"prefetch{prefetch}")
    delivered = []
    with connect() as connection:
        channel = connection.channel()
        channel.queue_declare(queue=queue_name)
        for number in range(8):
            channel.basic_publish(exchange="", routing_key=queue_name, body=str(number).encode())
        channel.basic_qos(prefetch_count=prefetch)
        tag = channel.basic_consume(
            queue=queue_name, auto_ack=False,
            on_message_callback=lambda ch, method, properties, body: delivered.append((method.delivery_tag, body)),
        )
        deadline = time.monotonic() + 2
        while time.monotonic() < deadline:
            connection.process_data_events(time_limit=0.1)
        count = len(delivered)
        assert count == prefetch, (prefetch, count)
        channel.basic_cancel(tag)
        for delivery_tag, body in delivered:
            channel.basic_ack(delivery_tag)
        channel.queue_delete(queue=queue_name)
    return {"prefetch": prefetch, "delivered_without_ack": count}

for value in (1, 4):
    print(observe_window(value))


## Step 2 — Start two workers before publishing

The next cell owns all worker threads and stops within a deadline. A ready signal ensures both consumers are registered before messages are published. The slow worker waits 80 ms per task; the fast worker waits 20 ms. Prefetch 1 prevents either from reserving many tasks at once.

Expected: every job completes once in this failure-free run. The faster worker often does more work, but exact distribution and timing depend on scheduling, so they are observations rather than correctness assertions. The brief sleeps model work; long blocking work requires careful heartbeat handling or a different execution design.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
import threading

queue_name = unique_name("competing")
job_count = 12
completed = []
lock = threading.Lock()
finished = threading.Event()
ready = {"slow": threading.Event(), "fast": threading.Event()}
with connect() as connection:
    connection.channel().queue_declare(queue=queue_name)

def worker(worker_name, delay):
    with connect() as connection:
        channel = connection.channel()
        channel.basic_qos(prefetch_count=1)
        def handle(ch, method, properties, body):
            time.sleep(delay)
            ch.basic_ack(method.delivery_tag)
            with lock:
                completed.append((worker_name, int(body)))
                if len(completed) == job_count:
                    finished.set()
        tag = channel.basic_consume(queue=queue_name, on_message_callback=handle, auto_ack=False)
        ready[worker_name].set()
        deadline = time.monotonic() + 12
        while not finished.is_set() and time.monotonic() < deadline:
            connection.process_data_events(time_limit=0.1)
        channel.basic_cancel(tag)
        if not finished.is_set():
            raise TimeoutError(f"Worker {worker_name} reached its deadline")

try:
    with ThreadPoolExecutor(max_workers=2) as pool:
        futures = [pool.submit(worker, "slow", 0.08), pool.submit(worker, "fast", 0.02)]
        if not all(signal.wait(timeout=5) for signal in ready.values()):
            finished.set()
            raise TimeoutError("Workers did not register in time")
        with connect() as connection:
            channel = connection.channel()
            channel.confirm_delivery()
            for number in range(job_count):
                channel.basic_publish(exchange="", routing_key=queue_name, body=str(number).encode(), mandatory=True)
        for future in futures:
            future.result(timeout=15)
    assert sorted(number for _, number in completed) == list(range(job_count))
    print("Completed jobs per worker:", dict(Counter(name for name, _ in completed)))
    print("Completion order:", completed)
finally:
    finished.set()
    with connect() as connection:
        connection.channel().queue_delete(queue=queue_name)


## Exercise — Tune and explain

Change the worker prefetch to 6, run Step 2 again, and compare assignment and completion order. Repeat three times. Then make both workers equally fast.

**Suggested interpretation:** larger prefetch can reduce waiting on network round trips, but lets a slow consumer reserve more work. Prefetch does not create worker parallelism. Competing workers also mean completion order can differ from publication order. Choose a value using measured latency, memory, task duration, and throughput; no universal value is best.

**Troubleshooting:** if a worker fails, inspect its future exception. Sharing one Pika connection across these threads is unsupported. `prefetch_count=0` removes the limit; do not use it as a generic performance fix. If interrupted, restart the kernel to stop abandoned threads and remove the named lesson queue in Management.

**Cleanup:** both experiments delete their own queues. No long-running worker remains after successful completion.

## References

- [RabbitMQ consumer prefetch](https://www.rabbitmq.com/docs/4.2/consumer-prefetch)
- [Pika FAQ on threads](https://pika.readthedocs.io/en/stable/faq.html)
